<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Allison/MLTestSuccess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Strengths to using LGBM Regressor:


*  It can run through large datasets very fast, purely numeric-heavy datasets (was able to run through ~4mill)
*   Optimized for numerical data

* If categorical features dominate, CatBoost may reduce preprocessing time and overfitting
* Not as memeory intensive, trained 4mill rows with very low RAM usage
* Does require one-hot encoding/target coding for the categorical variables.



In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb


drive.mount('/content/drive')
path = "/content/drive/MyDrive/model_df_clf_v2.feather"
data = pd.read_feather(path)


#Creating log-transformed LOS (target variable)
data['los_log'] = np.log1p(data['length_of_stay'])

#Categorical columns
categorical_cols = ['zip_code', 'health_service_area', 'facility_id', 'hospital_county', 'age_group',
                    'gender', 'race', 'ethnicity', 'admission_type', 'apr_mortality_risk',
                    'apr_severity_code', 'apr_drg_code', 'apr_mdc_code', 'ccsr_dx_code',
                    'payment_type', 'los_category']
#Featured columns
feature_cols = categorical_cols + ['num_payment_types']

#Convert categorical columns to category dtype
for col in categorical_cols:
  if col in data.columns:
    data[col] = data[col].astype('category')

# Convert numerical and drop missing data/zip codes
data['length_of_stay'] = pd.to_numeric(data['length_of_stay'], errors='coerce')
data = data.dropna(subset=['length_of_stay'])

if 'zip_code' in data.columns:
    data = data[data['zip_code'] != 'OOS']



# Define features and target
X = data[feature_cols]
y = data['los_log']

# LightGBM Regressor (memory-efficient and fast)
model = lgb.LGBMRegressor(n_estimators = 1200, learning_rate = 0.03, num_leaves = 63,
                          max_depth = -1, subsample = 0.8, colsample_bytree = 0.8,
                          n_jobs = -1, random_state = 42)
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)



# Evaluate Log Data
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.3f}")


Mounted at /content/drive
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.159161 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 982
[LightGBM] [Info] Number of data points in the train set: 3288528, number of used features: 17
[LightGBM] [Info] Start training from score 1.576022
MAE: 0.17
RMSE: 0.21
R2: 0.915


In [4]:
#Overfitting:

#Predict on Test & Training set
y_test_pred = model.predict(X_test)
y_train_pred = model.predict(X_train)

#Reverse log-transformation
y_test_pred_original = np.expm1(y_test_pred)
y_test_true = np.expm1(y_test)
y_train_pred_original = np.expm1(y_train_pred)
y_train_true = np.expm1(y_train)

#Calculate metrics for Test & Train sets
test_mae = mean_absolute_error(y_test_true, y_test_pred_original)
test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred_original))
test_r2 = r2_score(y_test_true, y_test_pred_original)

train_mae = mean_absolute_error(y_train_true, y_train_pred_original)
train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred_original))
train_r2 = r2_score(y_train_true, y_train_pred_original)

#Data Results
print("\nTraining Set:")
print(f"  MAE:  {train_mae:.3f} days")
print(f"  RMSE: {train_rmse:.3f} days")
print(f"  R²:   {train_r2:.3f}")

print("\nTest Set:")
print(f"  MAE:  {test_mae:.3f} days")
print(f"  RMSE: {test_rmse:.3f} days")
print(f"  R²:   {test_r2:.3f}")

#Calculate gaps of Test & Train
mae_gap = test_mae - train_mae
rmse_gap = test_rmse - train_rmse
r2_gap = train_r2 - test_r2

print("\nOverfitting Indicators:")
print(f"  MAE gap:  {mae_gap:+.3f} days ({(mae_gap/train_mae)*100:+.1f}%)")
print(f"  RMSE gap: {rmse_gap:+.3f} days ({(rmse_gap/train_rmse)*100:+.1f}%)")
print(f"  R² gap:   {r2_gap:+.3f} ({(r2_gap/train_r2)*100:+.1f}%)")

#Verdict
if mae_gap < 0.2 and r2_gap < 0.02:
    print("No Overfitting - Model generalizes well!")
elif mae_gap < 0.5 and r2_gap < 0.05:
    print("Minimal Overfitting - Acceptable for production")
elif mae_gap < 1.0 and r2_gap < 0.10:
    print("Moderate Overfitting - Consider regularization")
else:
    print("Significant Overfitting - Model needs adjustment")



Training Set:
  MAE:  1.475 days
  RMSE: 4.965 days
  R²:   0.676

Test Set:
  MAE:  1.530 days
  RMSE: 5.205 days
  R²:   0.647

Overfitting Indicators:
  MAE gap:  +0.055 days (+3.7%)
  RMSE gap: +0.240 days (+4.8%)
  R² gap:   +0.029 (+4.3%)
Minimal Overfitting - Acceptable for production


The Test set errors (MAE/RMSE) are very close to the Training set results:


*   MAE gap is only 0.055 days, ~4% difference which is low for LOS modeling
*   R2 only drops ~0.03 between Test and Training
* RMSE only increases 0.24 days

These numbers tell us the model is not memorizing the data, it is learning generalizable patterns within the data.

In healthcare LOS prediction:
* A MAE difference < 0.1 days is considered outstanding.
* A R² gap < 0.05 indicates stable generalization.
* RMSE differences below 0.5 days are typical for strong models.



In [3]:
#Saving Model
model.booster_.save_model("/content/drive/MyDrive/lightgbm_model.txt")